# Stage 2c -- per-residue protein embeddings (for the cross-attention head)

Every earlier stage collapsed each protein to **one vector**. A cross-attention head
needs the uncollapsed grid: **one row per residue**, so a ligand atom can attend to
residue 213 specifically.

Measured on our data, mean pooling was discarding about 9x the dimensionality:

| representation | effective rank |
|---|---|
| ProtBERT, mean-pooled | 12.7 / 1024 |
| ProtBERT, per-residue | 115.9 / 1024 |
| ESM-2 35M, per-residue | 150.8 / 480 |

**Before you run anything:** Runtime -> Change runtime type -> **L4 GPU**.

---

## Read this before running the full pass

The full ESM-2 650M output is **~9.8 GB**. That does not belong in Google Drive:
Colab Pro includes compute units, *not* storage, so Drive is your Google One quota
(15 GB on the free tier). Since the embedding pass only takes ~5 minutes,
**regenerating is cheaper than storing**.

The consequence is a sequencing constraint: the cache dies with the session, so the
full pass should run in the **same session** as the cross-attention training. That
code does not exist yet.

So this notebook has two halves:

1. **A validation pass on 512 proteins (~0.5 GB, ~1 min).** Proves the ragged-storage
   code at real scale on a real GPU, and measures ESM-2 650M's per-residue effective
   rank -- the open question about whether 650M carries richer per-residue structure
   than the smaller checkpoints. Run this now.
2. **The full 8,638-protein pass.** Leave it until the training code is ready.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.44,<6"
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda:", torch.cuda.is_available())

## 1. Upload

Drag in two files:

- `colab_upload/02_embed.py`  *(the updated one -- it now has `--save-residues`)*
- `data/processed/proteins.parquet`
- `data/processed/ligands.parquet`  *(only read for a row count, but the script expects it)*

In [ ]:
from google.colab import files
up = files.upload()
for n in up: print(" uploaded:", n)

## 2. Output goes to local disk, not Drive

`/content` is wiped when the runtime dies -- which is exactly right here. We are
treating the cache as disposable rather than paying Drive quota for something a
5-minute job can rebuild.

In [ ]:
import os, shutil
OUT = '/content/emb'
os.makedirs(OUT, exist_ok=True)
free_gb = shutil.disk_usage('/content').free / 1e9
print("writing to:", OUT)
print(f"free local disk: {free_gb:.0f} GB   (full pass needs ~10 GB)")

## 3. Validation pass -- 512 proteins

What to watch for in the output:

- `per-residue: N residues x 1280 @ fp16 = X GB`, and the `cap` it reports.
  The cap is `max_length - n_special`, so 1022 at `--prot-max 1024`.
- **`integrity: ... r = 1.000000`** -- this is the one that matters. It recomputes
  the mean pool *from the stored fp16 ragged array* and correlates it against the
  fp32 mean from the forward pass. One number that simultaneously validates the
  ragged layout, the offsets table, the special-token masking and the precision.
  Anything below 0.999 and the script raises instead of handing you a corrupt cache.

In [ ]:
# One line deliberately: backslash continuations inside an IPython `!` cell are a
# known source of silent mangling, so the command stays unbroken.
!python 02_embed.py --data . --out "$OUT" --save-residues --exclude-special --skip-ligands --prot-model facebook/esm2_t33_650M_UR50D --prot-max 1024 --tag _val650 --limit 512 --prot-budget 8192

## 4. Does 650M actually carry richer per-residue structure?

The open question. On the pooled representations, 650M and 150M scored identically
(0.7350 vs 0.7366) -- but that was measured in a regime where ~9x of the
dimensionality was being destroyed before the head saw anything, so it says nothing
about per-residue quality.

Two numbers below:

- **across all residues** -- how many directions the representation uses overall.
- **within a single protein** -- how much residues of the *same* protein differ from
  each other. This is the one that governs cross-attention, because attention can
  only select among things that differ. ESM-2 35M scored ~71 over 254 residues.

In [ ]:
import numpy as np, os

def eff_rank(X):
    X = np.asarray(X, dtype=np.float64)
    X = X - X.mean(0)
    s = np.linalg.svd(X, compute_uv=False)
    p = s**2 / max((s**2).sum(), 1e-300)
    return float(np.exp(-(p * np.log(p + 1e-300)).sum()))

R = np.load(f'{OUT}/prot_residues_val650.npy', mmap_mode='r')
off = np.load(f'{OUT}/prot_offsets_val650.npy')
print(f"residue grid {R.shape}   proteins {len(off)-1:,}")
print(f"residues per protein: min {np.diff(off).min()}  "
      f"median {int(np.median(np.diff(off)))}  max {np.diff(off).max()}")
print()

rng = np.random.default_rng(0)
sel = np.sort(rng.choice(R.shape[0], min(20000, R.shape[0]), replace=False))
print(f"  across all residues (n={len(sel):,})   eff.rank "
      f"{eff_rank(R[sel]):7.1f} / {R.shape[1]}")

lens = np.diff(off)
big = np.argsort(lens)[-8:]
within = [eff_rank(R[off[i]:off[i+1]]) for i in big if lens[i] > 50]
print(f"  within one protein, mean of {len(within)}  eff.rank "
      f"{np.mean(within):7.1f}   (ESM-2 35M scored ~71)")
print()
print("  pooled, for comparison:  ProtBERT mean 12.7/1024, ESM-2 650M mean 23.9/1280")

## 5. The full pass -- DO NOT RUN YET

Leave this until the cross-attention training code is ready, so the 9.8 GB cache and
the training happen in one session. Running it now just burns 5 minutes of L4 time
on something that will be deleted.

Expect ~5 minutes and a `NOTE:` line reporting ~7% of proteins truncated at 1024.

In [ ]:
# Uncomment when the cross-attention training code is ready to run in this session.
# !python 02_embed.py --data . --out "$OUT" --save-residues --exclude-special --skip-ligands --prot-model facebook/esm2_t33_650M_UR50D --prot-max 1024 --tag _esm650 --prot-budget 8192

## 6. What the cache looks like

```
prot_residues_esm650.npy   (3.81M x 1280) fp16   every residue, end to end
prot_offsets_esm650.npy    (8,639,)       int64  protein i = rows offsets[i]:offsets[i+1]
prot_pooler_esm650.npy     (8,638 x 1280) fp32   unused -- ESM's pooler is randomly initialised
prot_mean_esm650.npy       (8,638 x 1280) fp32   the baseline to beat (full R = 0.7350)
```

The ragged layout is why this is 9.8 GB instead of 79.4 GB: our proteins run from 9 to
7,182 residues with a median of 375, so a padded rectangular array would be 93%
padding. Padding happens per *batch* at training time instead, to the longest member
of that batch.